# Experiment: Product Review Sentiment EDA

Objective: perform a reproducible exploratory analysis for the product-review sentiment dataset,
inspect data quality, run a quick TF-IDF baseline, and export a notebook-generated submission.


## 1. Setup


In [ ]:
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
np.random.seed(42)
random.seed(42)


## 2. Configuration


In [ ]:
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root(Path.cwd()).resolve()
DATA_DIR = PROJECT_ROOT / "data" / "raw"
TRAIN_PATH = DATA_DIR / "train.json"
TEST_PATH = DATA_DIR / "test.json"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SUBMISSION_DIR = PROJECT_ROOT / "data" / "submissions"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR = PROJECT_ROOT / "models"

for target in [PROCESSED_DIR, SUBMISSION_DIR, REPORTS_DIR, MODELS_DIR]:
    target.mkdir(parents=True, exist_ok=True)

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

RANDOM_SEED = 42
PROJECT_ROOT


## 3. Data Loading


In [ ]:
from sentiment_project.data import LABEL_COL, REVIEW_COL, load_test_dataframe, load_train_dataframe

train_df = load_train_dataframe(TRAIN_PATH)
test_df = load_test_dataframe(TEST_PATH)

print(f"Train rows: {len(train_df):,}")
print(f"Test rows:  {len(test_df):,}")
train_df.head()


## 4. Data Validation


In [ ]:
validation_summary = {
    "train_missing_reviews": int(train_df[REVIEW_COL].isna().sum()),
    "train_missing_labels": int(train_df[LABEL_COL].isna().sum()),
    "test_missing_reviews": int(test_df[REVIEW_COL].isna().sum()),
    "train_duplicate_reviews": int(train_df.duplicated(subset=[REVIEW_COL]).sum()),
    "label_distribution": train_df[LABEL_COL].value_counts().to_dict(),
}

validation_summary


## 5. Exploratory Analysis


In [ ]:
train_df["review_char_len"] = train_df[REVIEW_COL].str.len()
train_df["review_word_len"] = train_df[REVIEW_COL].str.split().str.len()

display(train_df[["review_char_len", "review_word_len"]].describe(percentiles=[0.5, 0.75, 0.9, 0.95]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(data=train_df, x=LABEL_COL, ax=axes[0])
axes[0].set_title("Class distribution")
axes[0].set_xlabel("Sentiment")

sns.histplot(data=train_df, x="review_word_len", hue=LABEL_COL, bins=50, ax=axes[1], element="step")
axes[1].set_title("Review length distribution (words)")
axes[1].set_xlabel("Word count")

plt.tight_layout()
plt.show()


## 6. Feature Engineering


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_features=50_000,
    sublinear_tf=True,
)

x_tfidf = vectorizer.fit_transform(train_df[REVIEW_COL])
{
    "tfidf_shape": x_tfidf.shape,
    "vocab_size": len(vectorizer.vocabulary_),
    "avg_non_zero_per_doc": float(x_tfidf.getnnz(axis=1).mean()),
}


## 7. Modeling


In [ ]:
from sklearn.model_selection import train_test_split
from sentiment_project.modeling import build_tfidf_logreg_pipeline

x_train, x_val, y_train, y_val = train_test_split(
    train_df[REVIEW_COL],
    train_df[LABEL_COL],
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=train_df[LABEL_COL],
)

baseline_model = build_tfidf_logreg_pipeline(random_state=RANDOM_SEED)
baseline_model.fit(x_train, y_train)


## 8. Evaluation


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score

val_preds = baseline_model.predict(x_val)

metrics = {
    "accuracy": float(accuracy_score(y_val, val_preds)),
    "macro_f1": float(f1_score(y_val, val_preds, average="macro")),
}
metrics


In [ ]:
print(classification_report(y_val, val_preds, digits=4))
ConfusionMatrixDisplay.from_predictions(y_val, val_preds)
plt.title("Validation confusion matrix")
plt.show()


## 9. Inference / Submission


In [ ]:
from sentiment_project.inference import build_submission_dataframe, save_submission_csv

submission_path = SUBMISSION_DIR / "submission_eda_quick.csv"
test_preds = baseline_model.predict(test_df[REVIEW_COL])
submission_df = build_submission_dataframe(test_preds, include_id=True)
save_submission_csv(submission_df, submission_path)

print(f"Saved: {submission_path.relative_to(PROJECT_ROOT)}")
submission_df.head()


## 10. Conclusions / Next Steps

- The notebook verifies data quality, class balance, and review-length behavior.
- A deterministic TF-IDF baseline is trained and evaluated on a holdout split.
- A reproducible submission file is exported for assignment workflow integration.

Next:
1. Run hyperparameter sweeps for `C`, `ngram_range`, and `max_features`.
2. Add model-error inspection for false positives/false negatives.
3. Compare this baseline against a neural model (e.g., DistilBERT).
